# رفع ملفات إسناد — 6000 حكم
افتحي هذا النوتبوك بحساب Google الذي توجد فيه ملفات المشروع، ثم شغّلي الخلايا بالترتيب.
سيقرأ الملفات من `MyDrive/Esnad/final_artifacts` ويتحقق من عدد الأحكام، ثم يرفع الملفات الثلاثة في commit واحد إلى فرع `upscaling`.

- اسم حساب GitHub والتوكن يجب أن يكونا لنفس الحساب، مع صلاحية الكتابة في مستودع Esnad.
- أدخلي التوكن فقط في الخانة المخفية عند الطلب. استخدمي توكنًا جديدًا إذا سبق أن شاركتِ القديم.
- تبقى ملفات MOJ والنسخة السابقة من الفهرس كما هي.
- إذا رفض GitHub الرفع بسبب تغيّر الفرع أثناء التشغيل، أعيدي تشغيل خلية الرفع لتأخذ نسخة حديثة؛ لا تستخدمي force push.


## 1. ربط Google Drive
وافقي على ربط الحساب الذي توجد فيه الملفات الثلاثة.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. تثبيت مكتبات قراءة الملفات
هذه الخطوة لا تعيد إنشاء القضايا أو تحميل نموذج الذكاء الاصطناعي.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "pandas", "pyarrow"],
    check=True,
)


## 3. التحقق والرفع
شغّلي الخلية وأدخلي اسم حساب GitHub وإيميله، ثم التوكن في الخانة المخفية.
لن يتغير فرع `main`. نجاح الخطوة يظهر رسالة «تم رفع الملفات الثلاثة» مع رابط commit.
إذا لم تجدي الملفات، افتحي مجلد `Esnad/final_artifacts` في Drive أولًا وتحققي من وجودها.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tempfile
from getpass import getpass

SOURCE_DIR = Path("/content/drive/MyDrive/Esnad/final_artifacts")
REPOSITORY_URL = "https://github.com/raghad-cs/Esnad.git"
BRANCH = "upscaling"
FILES = [
    "alarb_judgments_6000.index",
    "alarb_judgments_metadata_6000.parquet",
    "alarb_search_config.json",
]

def publish_alarb():
    for name in FILES:
        path = SOURCE_DIR / name
        if not path.is_file():
            raise FileNotFoundError(f"الملف غير موجود: {path}")
        if path.stat().st_size >= 100 * 1024**2:
            raise ValueError(f"الملف يتجاوز حد Git العادي: {name}")
        print(name, f"{path.stat().st_size / 1024**2:.2f} MiB")

    with tempfile.TemporaryDirectory(prefix="esnad-upload-") as temporary_dir:
        temporary_dir = Path(temporary_dir)
        repo_dir = temporary_dir / "Esnad"
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH,
             REPOSITORY_URL, str(repo_dir)],
            check=True,
        )

        # Import the validator from the freshly cloned branch.
        sys.path.insert(0, str(repo_dir))
        try:
            from alarb_artifacts import load_alarb_artifacts
            index, metadata, config = load_alarb_artifacts(SOURCE_DIR)
            if config["number_of_cases"] != 6000:
                raise ValueError("ملف الإعدادات يجب أن يحدد 6000 حكم.")
            print(f"تم التحقق من {index.ntotal} حكم و{len(metadata)} صف بيانات.")

            target_dir = repo_dir / "artifacts" / "alarb"
            target_dir.mkdir(parents=True, exist_ok=True)
            for name in FILES:
                shutil.copy2(SOURCE_DIR / name, target_dir / name)
            load_alarb_artifacts(target_dir)
        finally:
            sys.path.remove(str(repo_dir))

        def git(*args, **kwargs):
            return subprocess.run(
                ["git", "-C", str(repo_dir), *args],
                check=True,
                **kwargs,
            )

        paths = [f"artifacts/alarb/{name}" for name in FILES]
        git("add", "--", *paths)
        unchanged = subprocess.run(
            ["git", "-C", str(repo_dir), "diff", "--cached", "--quiet"],
            check=False,
        ).returncode
        if unchanged == 0:
            print("الملفات الثلاثة موجودة بالفعل بهذه النسخة؛ لا يوجد شيء جديد لرفعه.")
            return
        if unchanged != 1:
            raise RuntimeError("تعذر فحص التغييرات.")

        staged_paths = git(
            "diff", "--cached", "--name-only", text=True, capture_output=True
        ).stdout.splitlines()
        if not set(staged_paths) <= set(paths):
            raise RuntimeError("توجد تغييرات خارج ملفات التحديث المحددة.")

        username = input("اسم حساب GitHub الذي سيرفع الملفات: ").strip()
        email = input("إيميل نفس الحساب أو عنوان noreply الخاص به: ").strip()
        if not username or not email:
            raise ValueError("أدخلي اسم الحساب والإيميل.")
        git("config", "user.name", username)
        git("config", "user.email", email)
        git("commit", "-m", "Publish verified 6000-judgment ALARB artifacts")

        # The helper contains no credential; it reads only the push process environment.
        askpass_path = temporary_dir / "askpass.py"
        askpass_path.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            "key = 'ESNAD_GITHUB_USER' if 'username' in sys.argv[1].lower() "
            "else 'ESNAD_GITHUB_TOKEN'\n"
            "print(os.environ[key])\n",
            encoding="utf-8",
        )
        askpass_path.chmod(0o700)
        token = getpass("التوكن الخاص بنفس حساب GitHub (إدخال مخفي): ").strip()
        if not token:
            raise ValueError("لم يتم إدخال توكن.")
        push_env = os.environ.copy()
        push_env.update({
            "GIT_ASKPASS": str(askpass_path),
            "GIT_TERMINAL_PROMPT": "0",
            "ESNAD_GITHUB_USER": username,
            "ESNAD_GITHUB_TOKEN": token,
        })
        try:
            # Disable inherited credential storage. No token is placed in the remote URL.
            git("-c", "credential.helper=", "push", "origin", f"HEAD:{BRANCH}",
                env=push_env)
        finally:
            token = ""
            push_env.pop("ESNAD_GITHUB_TOKEN", None)
            askpass_path.unlink(missing_ok=True)

        local_sha = git("rev-parse", "HEAD", text=True, capture_output=True).stdout.strip()
        remote_ref = git(
            "ls-remote", "origin", f"refs/heads/{BRANCH}",
            text=True, capture_output=True,
        ).stdout.split()
        if not remote_ref or remote_ref[0] != local_sha:
            raise RuntimeError("لم نتمكن من تأكيد آخر نسخة على GitHub؛ افحصي الفرع.")
        print("تم رفع ملفات 6000 حكم وربط المحرك بها على فرع upscaling.")
        print(f"https://github.com/raghad-cs/Esnad/commit/{local_sha}")

publish_alarb()

نوع التوكن يعتمد على الحساب:
- مالكة المستودع `raghad-cs`: توكن Fine-grained محدد لمستودع Esnad مع `Contents: Read and write`.
- المتعاونة من حساب آخر: استخدمي توكن Classic من حسابك بصلاحية `public_repo` للمستودع العام؛ توكن Fine-grained لا يدعم الكتابة في مستودع شخص آخر حاليًا.
اختاري مدة انتهاء قصيرة، واحذفي التوكن بعد انتهاء الرفع إذا لم تعودي بحاجة إليه.
[توثيق GitHub لأنواع التوكنات وحدودها](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens)
